# CMIP6 Data generation & merging 

CMIP6 data will be generated from 1995-2014, loaded and merged into one file

**Note: Depending on if you want to run historical, or an ssp, changes you need to make are:**
1. Cell 5: Select scenario
2. Cell 6: Define period

## Start-up

In [9]:
# general python
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

#niceties
from rich import print

import yaml

In [10]:
# General eWaterCycle
import ewatercycle
import ewatercycle.models
import ewatercycle.forcing

In [11]:
# Choose what you want to run:

Scenario = "historical"    # Options: historical, ssp119, ssp126, ssp245, ssp,370, ssp585

### Define range & forcing paths

In [12]:
# Defining period range

# For CMIP trial:
periods = [
    [1995, 2000, 2005, 2010],
    [1999, 2004, 2009, 2014]
]

folder_names = []
forcing_paths = {}

# Generate file names & paths
for i in range(len(periods[0])):
    start_year = periods[0][i]
    end_year = periods[1][i]
    
    folder_name = f'CMIP6-{start_year}-{end_year}'                        
    forcing_path = Path.home() / "BEP-maxime" / "Workyard" / "forcings" / "CMIP6" / Scenario / folder_name
    # forcing_path = Path.home() / "BEP-maxime" / "Workyard" / "forcings" / "CMIP6" / "SSP" / folder_name
    # forcing_path = Path.home() / "BEP-maxime" / "Workyard" / "forcings" / "CMIP6" / "SSP" / folder_name
    # forcing_path = Path.home() / "BEP-maxime" / "Workyard" / "forcings" / "CMIP6" / "SSP" / folder_name
    # forcing_path = Path.home() / "BEP-maxime" / "book" / "thesis_projects" / "BSc" / "2026_Q4_MaximedeBekker_CEG" / "Workyard" / "forcings" / folder_name

    folder_names.append(folder_name)
    forcing_paths[folder_name] = forcing_path

    forcing_path.mkdir(exist_ok=True, parents=True)

# Call path test
print(forcing_paths[folder_names[0]])

print(periods[0][1])

print(periods[1][-1])

/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-1995-1999

2000

2014

### Generate CMIP6 data

In [13]:
# Shapefile path
shape_file = Path.home() / "BEP-maxime" / "Workyard" / "Shapefiles" / "07DA001_basin.shp"
# shape_file = Path.home() / "BEP-maxime" / "book" / "thesis_projects" / "BSc" / "2026_Q4_MaximedeBekker_CEG" / "Workyard" / "Shapefiles" / "07DA001_basin.shp""

# Choose CMIP6 dataset
CMIP_dataset = {'dataset': 'MPI-ESM1-2-HR', 'project': 'CMIP6', 'grid' : 'gn', 'exp': Scenario, 'ensemble': 'r1i1p1f1'}


# Generate forcing data
# for i in range(1):
for i in range(len(folder_names)):
    # Change start year to start time so ERA5 can interpret it
    start_time_CMIP = f'{periods[0][i]}-01-01T00:00:00Z'
    end_time_CMIP = f'{periods[1][i]}-12-31T00:00:00Z'

    # Skip if folder is not empty to prevent errors
    if any(forcing_paths[folder_names[i]].iterdir()):
        print(f"Skipping {folder_names[i]} because the folder is not empty.")
        continue

    # Load forcings
    CMIP_forcing = ewatercycle.forcing.sources['LumpedMakkinkForcing'].generate(
        dataset=CMIP_dataset,
        start_time=start_time_CMIP,
        end_time=end_time_CMIP,
        shape=shape_file,
        directory=forcing_paths[folder_names[i]]
    )

    # I need info because my patience is limited
    print(f"Finished {folder_names[i]}")

Skipping CMIP6-1995-1999 because the folder is not empty.

Skipping CMIP6-2000-2004 because the folder is not empty.

Skipping CMIP6-2005-2009 because the folder is not empty.

Skipping CMIP6-2010-2014 because the folder is not empty.

### Load CMIP6 data

In [15]:
CMIP_forcings = {}

for i in range(len(folder_names)):
    # Create new path to find relevant files in the folder
    directory_new = forcing_paths[folder_names[i]] / "work" / "diagnostic" / "script"
    
    # Load forcings
    CMIP_forcings[folder_names[i]] = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=directory_new)

# Test to see if it works
print(folder_names)
CMIP_forcings[folder_names[0]]

['CMIP6-1995-1999', 'CMIP6-2000-2004', 'CMIP6-2005-2009', 'CMIP6-2010-2014']

LumpedMakkinkForcing(start_time='1995-01-01T00:00:00Z', end_time='1999-12-31T00:00:00Z', directory=PosixPath('/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-1995-1999/work/diagnostic/script'), shape=PosixPath('/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-1995-1999/work/diagnostic/script/07DA001_basin.shp'), filenames={'pr': 'CMIP6_MPI-ESM1-2-HR_day_historical_r1i1p1f1_pr_gn_1995-1999.nc', 'tas': 'CMIP6_MPI-ESM1-2-HR_day_historical_r1i1p1f1_tas_gn_1995-1999.nc', 'rsds': 'CMIP6_MPI-ESM1-2-HR_day_historical_r1i1p1f1_rsds_gn_1995-1999.nc', 'evspsblpot': 'Derived_Makkink_evspsblpot.nc'})

### Merge CMIP6 data

In [16]:
# Function to seperate files based on file names and combine same file names from different folders
def combine_variable(var_name):
    datasets = []

    for i in range(len(folder_names)):

        if var_name == "evspsblpot":
            file_name = "Derived_Makkink_evspsblpot.nc"
        else:
            file_name = f"CMIP6_MPI-ESM1-2-HR_day_{Scenario}_r1i1p1f1_{var_name}_gn_{periods[0][i]}-{periods[1][i]}.nc"

        directory = forcing_paths[folder_names[i]] / "work" / "diagnostic" / "script" / file_name

        print(f"Loading {directory}")

        datasets.append(xr.open_dataset(directory))

    combined = xr.concat(datasets, dim="time")
    return combined

In [17]:
# Running function for various file name types
combined_variables = {}
var_names = ["pr", "tas", "rsds", "evspsblpot"]

combined_path = Path.home() / "BEP-maxime" / "Workyard" / "forcings" / "CMIP6" / Scenario / f'CMIP6-{periods[0][0]}-{periods[1][-1]}'
# combined_path = Path.home() / "BEP-maxime" / "book" / "thesis_projects" / "BSc" / "2026_Q4_MaximedeBekker_CEG" / "Workyard" / "forcings" / f'ERA5-{periods[0][0]}-{periods[1][-1]}'
combined_path.mkdir(parents=True, exist_ok=True)

for i in range(len(var_names)):
    combined_variables[var_names[i]] = combine_variable(var_names[i])
    combined_variables[var_names[i]].to_netcdf(combined_path / f'combined_ERA5_{periods[0][0]}_{periods[1][-1]}_{var_names[i]}.nc')

Loading 
/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-1995-1999/work/diagnostic/script/CMIP6_MPI-ESM1-2-
HR_day_historical_r1i1p1f1_pr_gn_1995-1999.nc

Loading 
/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-2000-2004/work/diagnostic/script/CMIP6_MPI-ESM1-2-
HR_day_historical_r1i1p1f1_pr_gn_2000-2004.nc

Loading 
/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-2005-2009/work/diagnostic/script/CMIP6_MPI-ESM1-2-
HR_day_historical_r1i1p1f1_pr_gn_2005-2009.nc

Loading 
/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-2010-2014/work/diagnostic/script/CMIP6_MPI-ESM1-2-
HR_day_historical_r1i1p1f1_pr_gn_2010-2014.nc

Loading 
/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-1995-1999/work/diagnostic/script/CMIP6_MPI-ESM1-2-
HR_day_historical_r1i1p1f1_tas_gn_1995-1999.nc

...

Loading 
/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-2010-2014/work/diagnostic/script/CMIP6_MPI-ESM1-2-
HR_day_historical_r1i1p1f1_rsds_gn_2010-2014.nc

Loading 
/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-1995-1999/work/diagnostic/script/Derived_Makkink_e
vspsblpot.nc

Loading 
/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-2000-2004/work/diagnostic/script/Derived_Makkink_e
vspsblpot.nc

Loading 
/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-2005-2009/work/diagnostic/script/Derived_Makkink_e
vspsblpot.nc

Loading 
/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-2010-2014/work/diagnostic/script/Derived_Makkink_e
vspsblpot.nc

### Create YAML file

In [18]:
# Create yaml
forcing_yaml = {
    "start_time": f"{periods[0][0]}-01-01T00:00:00Z",
    "end_time": f"{periods[1][-1]}-12-31T00:00:00Z",
    "shape": str(shape_file),
    "filenames": {
        "pr": f"combined_CMIP6_{periods[0][0]}_{periods[1][-1]}_pr.nc",
        "tas": f"combined_CMIP6_{periods[0][0]}_{periods[1][-1]}_tas.nc",
        "rsds": f"combined_CMIP6_{periods[0][0]}_{periods[1][-1]}_rsds.nc",
        "evspsblpot": f"combined_CMIP6_{periods[0][0]}_{periods[1][-1]}_evspsblpot.nc"
    }}

# Save the YAML file
yaml_file_path = combined_path / "ewatercycle_forcing.yaml"

with open(yaml_file_path, "w") as yaml_file:
    yaml.dump(forcing_yaml, yaml_file, default_flow_style=False)

In [19]:
CMIP6_combined_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=combined_path)
print(CMIP6_combined_forcing)

LumpedMakkinkForcing(
    start_time='1995-01-01T00:00:00Z',
    end_time='2014-12-31T00:00:00Z',
    directory=PosixPath('/home/maxime/BEP-maxime/Workyard/forcings/CMIP6/historical/CMIP6-1995-2014'),
    shape=PosixPath('/home/maxime/BEP-maxime/Workyard/Shapefiles/07DA001_basin.shp'),
    filenames={
        'evspsblpot': 'combined_CMIP6_1995_2014_evspsblpot.nc',
        'pr': 'combined_CMIP6_1995_2014_pr.nc',
        'rsds': 'combined_CMIP6_1995_2014_rsds.nc',
        'tas': 'combined_CMIP6_1995_2014_tas.nc'
    }
)